# Chapter 12: Cross-Validation and Model Evaluation Techniques

## Ringkasan Bab & Penjelasan Teoretis
Evaluasi pada satu pembagian `train_test_split` saja dapat memberikan estimasi performa yang bias akibat kebetulan pengacakan sampel. **Cross-Validation (Validasi Silang)** membagi dataset secara sistematis ke dalam beberapa lipatan (*folds*) untuk menguji kemampuan generalisasi model secara menyeluruh.

### 1. Metode Cross-Validation Dasar & Lanjutan
* **K-Fold Cross-Validation (`KFold`)**: Membagi dataset menjadi $k$ lipatan berukuran sama; model dilatih pada $k-1$ lipatan dan diuji pada 1 lipatan sisanya secara bergantian sebanyak $k$ kali.
* **Stratified K-Fold (`StratifiedKFold`)**: Memastikan proporsi distribusi kelas target pada setiap lipatan sama persis dengan proporsi pada dataset utuh. Wajib digunakan pada dataset klasifikasi yang timpang (*imbalanced*).
* **TimeSeriesSplit (`TimeSeriesSplit`)**: Validasi silang khusus data runtun waktu di mana himpunan latih selalu berada di masa lalu sebelum himpunan uji (*expanding window*) guna mencegah kebocoran informasi masa depan.

### 2. Evaluasi Generalisasi dengan *Learning Curves* & *Validation Curves*
* **Learning Curve**: Memplot skor pelatihan dan validasi terhadap jumlah sampel latih untuk mendiagnosis apakah penambahan data dapat mengatasi *overfitting* (celah lebar antara kurva latih dan validasi).
* **Validation Curve**: Memplot skor pelatihan dan validasi terhadap variasi satu hyperparameter spesifik (seperti `C` atau `max_depth`) untuk menemukan titik optimal sebelum model mengalami *overfitting*.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (
    KFold, StratifiedKFold, TimeSeriesSplit,
    cross_val_score, learning_curve, validation_curve, GridSearchCV
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

data = load_breast_cancer()
X, y = data.data, data.target
pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))

# 1. Perbandingan KFold vs StratifiedKFold
kf = KFold(n_splits=5, shuffle=True, random_state=2024)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=2024)

scores_kf = cross_val_score(pipe, X, y, cv=kf, scoring='accuracy')
scores_skf = cross_val_score(pipe, X, y, cv=skf, scoring='accuracy')

print(f"5-Fold CV Accuracy: Mean = {scores_kf.mean():.4f}, Std = {scores_kf.std():.4f}")
print(f"Stratified 5-Fold CV Accuracy: Mean = {scores_skf.mean():.4f}, Std = {scores_skf.std():.4f}")

# 2. Validation Curve (Menilai Pengaruh Hyperparameter C pada LogisticRegression)
param_range = np.logspace(-4, 3, 8)
train_scores, val_scores = validation_curve(
    pipe, X, y, param_name="logisticregression__C", param_range=param_range, cv=skf, scoring="accuracy"
)

train_mean = np.mean(train_scores, axis=1)
val_mean = np.mean(val_scores, axis=1)

# 3. Learning Curve
train_sizes, lc_train, lc_val = learning_curve(
    pipe, X, y, train_sizes=np.linspace(0.1, 1.0, 8), cv=skf, scoring="accuracy"
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
ax1.plot(param_range, train_mean, label="Training Score", marker='o')
ax1.plot(param_range, val_mean, label="Cross-Validation Score", marker='s')
ax1.set_xscale("log")
ax1.set_xlabel("Parameter C (LogisticRegression)")
ax1.set_ylabel("Accuracy")
ax1.set_title("Validation Curve")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(train_sizes, np.mean(lc_train, axis=1), label="Training Score", marker='o')
ax2.plot(train_sizes, np.mean(lc_val, axis=1), label="Cross-Validation Score", marker='s')
ax2.set_xlabel("Number of Training Examples")
ax2.set_ylabel("Accuracy")
ax2.set_title("Learning Curve")
ax2.legend()
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()